# DiploMind: 3×3 dataset generation (language × depth), v2

Every article is processed. There are no topic filters and no quality rejects.

Each raw Urdu article becomes one English summary, then 9 cells:
- **depths:** easy, medium, hard (English written first, from the summary)
- **languages:** `en`, `urdish`, `ur` (Urdish and Urdu are conversions of the English cell at the same depth)

Urdish = Urdu written in English (Roman) letters with English words mixed in, the way people text in Pakistan.

**What changed from v1:**
- No IR / sensitive filter and no quality checks: every article with text is processed and kept.
- The gold examples are sent as separate chat turns, so the model no longer copies them into its answer.
- Every depth must keep every fact (v2 first try shrank 'hard' to one sentence).
- A separate, stronger model (`LANG_MODEL`) writes the Urdu and Urdish, and the Urdu translation sees the original article for correct names.
- Numbers are kept as written (v1 turned "September" into 9).
- Output files are named `*_v2_*`, so v1 results don't get in the way.

**3 machines:** set a different `MACHINE_ID` on each (section 3). Re-running the run cell resumes where it stopped.

## 1. Install

In [39]:
!pip install -q openai

## 2. API key
Paste your W&B key below. Clear this cell before sharing the notebook.
On Kaggle you can leave it empty and add a secret named `WANDB_API_KEY` instead (Add-ons → Secrets).

In [40]:
WANDB_API_KEY = ""   # paste your key here
WANDB_PROJECT = "m-ibrahim094131-fast-nuces/test"

import os
if not WANDB_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception:
        WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")
print("API key set" if WANDB_API_KEY else "No API key yet: paste it above")

API key set


## 3. Settings

In [41]:
import os, re, csv, json, time, glob, random, hashlib, threading, collections
from concurrent.futures import ThreadPoolExecutor

MACHINE_ID   = "bisma_test"            # m1 / m2 / m3: different on every machine
ARTICLES_DIR = "/kaggle/input/datasets/bismaamir0659/bbc-urdu-articles"      # folder with the scraped articles, e.g. /kaggle/input/<dataset>/articles_m1
OUT_ROOT     = "dataset_out"   # on Kaggle: /kaggle/working/dataset_out

MODEL = "deepseek-ai/DeepSeek-V3.1"  # summary + English easy/medium/hard
LANG_MODEL = "deepseek-ai/DeepSeek-V3.1"           # English -> Urdish / Urdu (Llama 3.3 is weak at Urdu)
# Other W&B options worth trying for LANG_MODEL (see the comparison cell in section 10):
#   "google/gemma-4-31B-it", "moonshotai/Kimi-K2.6", "Qwen/Qwen3.8-27B"
BASE_URL = "https://api.inference.wandb.ai/v1"

LANGS  = ["en", "urdish", "ur"]
DEPTHS = ["easy", "medium", "hard"]

MAX_ARTICLES = 40   # None = every article in the folder; e.g. 10 for a test run
WORKERS      = 4      # articles processed in parallel; lower it if you hit rate limits
EXAMPLE_RATE = 0.3    # share of articles whose EASY cells get one everyday example
TEMPERATURE  = 0.3
URDISH_FROM_UR = False   # E5 test mode (only used when BASE = "en")
BASE     = "ur"                       # "ur": Urdu summary -> Urdu depths -> English + Urdish | "en": previous pipeline
UR_MODEL = "deepseek-ai/DeepSeek-V4.1-Flash"    # E6a; then try Kimi / DeepSeek V4.1-Flash     # model for the whole Urdu-based pipeline (E6b: "deepseek-ai/DeepSeek-V4.1-Flash")
TR_MODEL = "deepseek-ai/DeepSeek-V3.1"   # Urdu -> English and Urdish (conversion only; lighter and faster)
# Optional automatic split across machines. Leave NUM_SHARDS = 1 if each machine has its own folder.
NUM_SHARDS  = 1
SHARD_INDEX = 0

OUT_DIR = os.path.join(OUT_ROOT, MACHINE_ID)
os.makedirs(OUT_DIR, exist_ok=True)
RAW_PATH  = os.path.join(OUT_DIR, f"grid_raw_v2_{MACHINE_ID}.jsonl")    # every processed article (resume file)
if BASE == "en":
    PREP_PATH = os.path.join(OUT_DIR, f"prep_cache_v4_{MACHINE_ID}.jsonl")
else:   # separate cache per model, so switching models makes fresh Urdu summaries
    PREP_PATH = os.path.join(OUT_DIR, f"prep_cache_facts2_{UR_MODEL.split('/')[-1]}_{MACHINE_ID}.jsonl")
print("Machine:", MACHINE_ID, "| writing to", OUT_DIR)

Machine: bisma_test | writing to dataset_out/bisma_test


## 4. Load the model (W&B Inference client) and test it

In [42]:
from openai import OpenAI

assert WANDB_API_KEY, "Paste the API key in section 2 first."
client = OpenAI(base_url=BASE_URL, api_key=WANDB_API_KEY, project=WANDB_PROJECT)
def is_reasoning(m):   # reasoning models need a bigger token budget
    return any(k in m.lower() for k in ("gpt-oss", "kimi", "glm", "minimax", "qwen3", "deepseek-v4"))

for m in (MODEL, LANG_MODEL, UR_MODEL):
    r = client.chat.completions.create(model=m, max_tokens=200 if is_reasoning(m) else 5,
                                       messages=[{"role": "user", "content": "Reply with OK."}])
    print(f"{m}: {r.choices[0].message.content!r}")

deepseek-ai/DeepSeek-V3.1: 'OK.'
deepseek-ai/DeepSeek-V3.1: 'OK.'
deepseek-ai/DeepSeek-V4.1-Flash: 'OK'


## 5. Gold samples (few-shot examples)
Replace these with your own checked samples. They are sent to the model as earlier chat turns (example question → example answer), so they guide the style without ending up in the output.
- `GOLD_SOURCES`: short English texts, like the summaries the pipeline makes.
- `GOLD_DEPTH`: the same texts rewritten at each depth, **in the same order** as `GOLD_SOURCES`.
- `GOLD_EASY_META`: for each easy rewrite, the terms it explains and whether it has an everyday example (`yes`/`no`).
- `GOLD_LANG`: (English, converted) pairs for Urdish and Urdu.

In [43]:
GOLD_SOURCES = [
    "The UN Security Council has passed a resolution calling on the parties for an immediate ceasefire. Thirteen members voted in favour and two did not vote. Diplomatic experts say its impact depends on how strongly major powers support it.",
    "A new round of trade talks between the US and China begins next week. Both countries will try to narrow differences over tariffs and export restrictions. Observers say a major breakthrough is unlikely, though continued talks are a positive sign.",
    "The European Union has announced new economic sanctions on a country. Assets of some of its banks will be frozen and energy exports will be limited. The affected country called the decision unfair and threatened countermeasures.",
]

GOLD_DEPTH = {
    "easy": [
        "The UN Security Council has passed a resolution. A resolution is a formal decision. The Security Council is the part of the United Nations that is responsible for keeping peace in the world. The resolution asks the sides in a conflict to stop fighting immediately. This is called a ceasefire. Thirteen members voted for it, and two did not vote. Experts on diplomacy, which means how countries deal with each other, say the decision will only work if the major powers support it. This is because the Security Council cannot easily force countries to follow its decisions.",
        "The US and China will start a new round of trade talks next week. In trade talks, countries discuss the rules for buying and selling goods with each other. The two countries disagree about tariffs. Tariffs are extra taxes on goods that come from other countries. They also disagree about export restrictions, which are limits on what a country can sell abroad. For example, if a tariff makes imported chocolate more expensive, shoppers may buy local chocolate instead. Experts do not expect a big agreement. But they say it is a good sign that the two sides keep talking.",
        "The European Union has announced new economic sanctions on a country. The European Union is a group of European countries that make many decisions together. Sanctions are penalties on trade and money that are used to pressure a government. Some of the country's banks will have their assets frozen. This means the banks cannot use that money. The country's energy exports, meaning the energy it sells abroad, will also be limited. The country called the decision unfair. It also threatened countermeasures, which means it may take steps to hit back.",
    ],
    "medium": [
        "The UN Security Council, the UN body responsible for international peace and security, has passed a resolution calling on the parties for an immediate ceasefire. Thirteen members voted in favour and two did not vote. Diplomatic experts say the resolution's impact will depend on how strongly the major powers support it.",
        "A new round of trade talks between the United States and China begins next week. Both countries will try to narrow their differences over tariffs, which are taxes on imported goods, and over export restrictions. Observers say a major breakthrough is unlikely, but they see the continuation of the talks as a positive sign.",
        "The European Union has announced new economic sanctions, meaning penalties on trade and finance, against a country. The assets of some of its banks will be frozen and its energy exports will be limited. The affected country called the decision unfair and threatened countermeasures.",
    ],
    "hard": [
        "The UN Security Council has adopted a resolution calling on the parties to an immediate ceasefire, with thirteen members voting in favour and two not voting. Diplomatic experts say its impact will depend on how strongly the major powers back it.",
        "A new round of US-China trade talks begins next week, aimed at narrowing differences over tariffs and export restrictions. Observers consider a major breakthrough unlikely, but read the continuation of the dialogue itself as a positive signal.",
        "The European Union has announced a new package of economic sanctions against a country, a form of coercive economic pressure that freezes the assets of some of its banks and limits its energy exports. The targeted government called the decision unfair and threatened countermeasures.",
    ],
}

GOLD_EASY_META = [
    ("Security Council, resolution, ceasefire", "no"),
    ("trade talks, tariffs, export restrictions", "yes"),
    ("sanctions, frozen assets, countermeasures", "no"),
]

GOLD_LANG = {
        "urdish": [
        ("The UN Security Council has passed a resolution calling on the parties for an immediate ceasefire. Thirteen members voted in favour and two did not vote. Diplomatic experts say its impact depends on how strongly major powers support it.",
         "UN Security Council ne ek resolution pass ki hai jis mein parties se foran ceasefire ka mutalba kiya gaya hai. Resolution ke haq mein terah votes aaye jabke do members ne voting mein hissa nahi liya. Diplomatic experts ke mutabiq is resolution par amal hona is baat par depend karta hai ke bari powers ise kitna support karti hain."),
        ("The EU announced new sanctions on a country. Sanctions are penalties on trade and money, used to pressure a government. Some of the country's banks will have their assets frozen, which means the banks cannot use that money. Energy exports, such as oil and gas, will also be limited. The country called this unfair and threatened to respond.",
         "European Union ne ek mulk par nayi sanctions lagayi hain. Sanctions un pabandiyon ko kehte hain jo paise aur business par lagayi jati hain taake kisi hukumat par dabao dala jaye. Is mulk ke kuch banks ke assets freeze kiye jayein ge, yaani woh banks woh paisa istemal nahi kar sakein ge. Tail aur gas jaisi energy ki exports par bhi hadd lagegi. Us mulk ne kaha ke yeh na-insafi hai aur jawab dene ki dhamki di hai."),
        ("The UN Security Council is the UN group that decides on peace and security. It passed a resolution, which is a formal decision, asking both sides to stop fighting right away. This is called a ceasefire. Thirteen members voted yes and two chose not to vote. Experts say the decision will only work if the big powers support it.",
         "UN Security Council, UN ka woh group hai jo peace aur security ke faislay karta hai. Is ne ek resolution pass ki hai, yaani ek formal faisla, jis mein dono sides se kaha gaya hai ke foran larai band karein. Isay ceasefire kehte hain. Terah members ne haan mein vote diya aur do ne vote hi nahi diya. Experts kehte hain ke yeh faisla tabhi kaam karega jab bari powers iska saath dein."),
        ("The US and China will start a new round of trade talks next week. They disagree about tariffs, which are extra taxes on goods brought in from other countries. They also disagree about export restrictions, which are limits on what a country can sell abroad. For example, if a tariff makes imported chocolate cost more, shoppers may buy local chocolate instead. Experts do not expect a big result. Still, they say it is a good sign that the talks continue.",
         "US aur China agle hafte trade talks ka naya round shuru kar rahe hain. Dono tariffs par agree nahi karte, tariffs yaani woh extra tax jo bahar se aane wali cheezon par lagta hai. Export restrictions par bhi dono ka ikhtilaf hai, yaani yeh limits ke koi mulk bahar kya kuch bech sakta hai. Maslan, agar tariff ki wajah se imported chocolate mehngi ho jaye, to log local chocolate lena shuru kar dete hain. Experts ko kisi bare result ki umeed nahi hai. Phir bhi woh kehte hain ke talks ka chalte rehna achi baat hai."),
    ],
    "ur": [
        ("The UN Security Council has passed a resolution calling on the parties for an immediate ceasefire. Thirteen members voted in favour and two did not vote. Diplomatic experts say its impact depends on how strongly major powers support it.",
         "اقوام متحدہ کی سلامتی کونسل نے ایک قرارداد منظور کی ہے جس میں فریقین سے فوری جنگ بندی کا مطالبہ کیا گیا ہے۔ قرارداد کے حق میں تیرہ ووٹ آئے جبکہ دو ارکان نے ووٹنگ میں حصہ نہیں لیا۔ سفارتی ماہرین کے مطابق اس قرارداد پر عمل درآمد کا انحصار اس بات پر ہے کہ بڑی طاقتیں اس کی کتنی حمایت کرتی ہیں۔"),
        ("The UN Security Council is the UN group that decides on peace and security. It passed a resolution, which is a formal decision, asking both sides to stop fighting right away. This is called a ceasefire. Thirteen members voted yes and two chose not to vote. Experts say the decision will only work if the big powers support it.",
         "اقوام متحدہ کی سلامتی کونسل اقوام متحدہ کا وہ ادارہ ہے جو امن اور سلامتی کے فیصلے کرتا ہے۔ اس نے ایک قرارداد منظور کی، یعنی ایک باقاعدہ فیصلہ، جس میں دونوں فریقوں سے کہا گیا کہ فوراً لڑائی بند کریں۔ اسے جنگ بندی کہتے ہیں۔ تیرہ ارکان نے حق میں ووٹ دیا اور دو نے ووٹ نہیں دیا۔ ماہرین کا کہنا ہے کہ یہ فیصلہ تب ہی کارگر ہوگا جب بڑی طاقتیں اس کی حمایت کریں گی۔"),
    ],
}


GOLD_SOURCES_UR = [
    "اقوام متحدہ کی سلامتی کونسل نے ایک قرارداد منظور کی ہے جس میں فریقین سے فوری جنگ بندی کا مطالبہ کیا گیا ہے۔ تیرہ ارکان نے حق میں ووٹ دیا جبکہ دو نے ووٹ نہیں دیا۔ سفارتی ماہرین کا کہنا ہے کہ اس کا اثر اس بات پر منحصر ہے کہ بڑی طاقتیں اس کی کتنی حمایت کرتی ہیں۔",
    "امریکہ اور چین کے درمیان تجارتی مذاکرات کا نیا دور اگلے ہفتے شروع ہو رہا ہے۔ دونوں ممالک محصولات اور برآمدی پابندیوں پر اختلافات کم کرنے کی کوشش کریں گے۔ مبصرین کا کہنا ہے کہ کسی بڑی پیش رفت کا امکان کم ہے، تاہم مذاکرات کا جاری رہنا ایک مثبت اشارہ ہے۔",
    "یورپی یونین نے ایک ملک پر نئی اقتصادی پابندیوں کا اعلان کیا ہے۔ اس کے کچھ بینکوں کے اثاثے منجمد کیے جائیں گے اور توانائی کی برآمدات محدود کی جائیں گی۔ متاثرہ ملک نے اس فیصلے کو ناانصافی قرار دیا اور جوابی اقدامات کی دھمکی دی۔",
]

GOLD_DEPTH_UR = {
    "easy": [
        "اقوام متحدہ کی سلامتی کونسل نے ایک قرارداد منظور کی ہے۔ قرارداد ایک باقاعدہ فیصلہ ہوتا ہے۔ سلامتی کونسل اقوام متحدہ کا وہ حصہ ہے جس کا کام دنیا میں امن قائم رکھنا ہے۔ اس قرارداد میں لڑائی کے فریقوں سے کہا گیا ہے کہ وہ فوراً لڑائی بند کریں۔ اسے جنگ بندی کہتے ہیں۔ تیرہ ارکان نے اس کے حق میں ووٹ دیا اور دو نے ووٹ نہیں دیا۔ سفارت کاری، یعنی ملکوں کے آپس کے معاملات، کے ماہرین کہتے ہیں کہ یہ فیصلہ تب ہی کام کرے گا جب بڑی طاقتیں اس کا ساتھ دیں۔",
        "امریکہ اور چین اگلے ہفتے تجارتی بات چیت کا نیا دور شروع کریں گے۔ تجارتی بات چیت میں ملک آپس میں چیزیں خریدنے اور بیچنے کے اصولوں پر بات کرتے ہیں۔ دونوں ملکوں کا محصولات پر اختلاف ہے۔ محصولات وہ اضافی ٹیکس ہیں جو دوسرے ملکوں سے آنے والی چیزوں پر لگتے ہیں۔ ان کا برآمدی پابندیوں پر بھی اختلاف ہے، یعنی اس بات کی حد کہ کوئی ملک باہر کیا بیچ سکتا ہے۔ مثال کے طور پر، اگر محصول کی وجہ سے باہر سے آنے والی چاکلیٹ مہنگی ہو جائے تو لوگ مقامی چاکلیٹ خریدنے لگتے ہیں۔ ماہرین کسی بڑے معاہدے کی توقع نہیں کرتے۔ لیکن ان کا کہنا ہے کہ دونوں ملکوں کا بات چیت جاری رکھنا اچھی بات ہے۔",
        "یورپی یونین نے ایک ملک پر نئی اقتصادی پابندیوں کا اعلان کیا ہے۔ یورپی یونین یورپ کے ملکوں کا ایک گروپ ہے جو بہت سے فیصلے مل کر کرتے ہیں۔ پابندیاں تجارت اور پیسے پر لگائی جانے والی سزائیں ہیں جن سے کسی حکومت پر دباؤ ڈالا جاتا ہے۔ اس ملک کے کچھ بینکوں کے اثاثے منجمد کیے جائیں گے۔ اس کا مطلب ہے کہ یہ بینک وہ پیسہ استعمال نہیں کر سکیں گے۔ اس ملک کی توانائی کی برآمدات، یعنی وہ توانائی جو وہ باہر بیچتا ہے، بھی محدود کی جائیں گی۔ اس ملک نے فیصلے کو ناانصافی کہا۔ اس نے جوابی اقدامات کی دھمکی بھی دی، یعنی وہ بدلے میں اپنے قدم اٹھا سکتا ہے۔",
    ],
    "medium": [
        "اقوام متحدہ کی سلامتی کونسل، جو دنیا میں امن اور سلامتی کی ذمہ دار ہے، نے ایک قرارداد منظور کی ہے جس میں فریقین سے فوری جنگ بندی کا کہا گیا ہے۔ تیرہ ارکان نے حق میں ووٹ دیا اور دو نے ووٹ نہیں دیا۔ سفارتی ماہرین کہتے ہیں کہ اس کا اثر اس پر منحصر ہوگا کہ بڑی طاقتیں اس کی کتنی حمایت کرتی ہیں۔",
        "امریکہ اور چین کے درمیان تجارتی مذاکرات کا نیا دور اگلے ہفتے شروع ہوگا۔ دونوں ملک محصولات، یعنی باہر سے آنے والی چیزوں پر ٹیکس، اور برآمدی پابندیوں پر اپنے اختلافات کم کرنے کی کوشش کریں گے۔ مبصرین کو کسی بڑی پیش رفت کی امید نہیں، لیکن وہ مذاکرات کے جاری رہنے کو اچھی علامت سمجھتے ہیں۔",
        "یورپی یونین نے ایک ملک پر نئی اقتصادی پابندیوں کا اعلان کیا ہے، یعنی تجارت اور پیسے کے لین دین پر سزائیں۔ اس کے کچھ بینکوں کے اثاثے منجمد کیے جائیں گے اور اس کی توانائی کی برآمدات محدود ہوں گی۔ متاثرہ ملک نے اس فیصلے کو ناانصافی کہا اور جوابی اقدامات کی دھمکی دی۔",
    ],
    "hard": [
        "اقوام متحدہ کی سلامتی کونسل نے فریقین سے فوری جنگ بندی کا مطالبہ کرنے والی قرارداد منظور کر لی ہے۔ تیرہ ارکان نے حق میں ووٹ دیا اور دو نے ووٹ نہیں دیا۔ سفارتی ماہرین کے مطابق اس کا اثر اس پر منحصر ہوگا کہ بڑی طاقتیں اس کی کتنی حمایت کرتی ہیں۔",
        "امریکہ اور چین کے تجارتی مذاکرات کا نیا دور اگلے ہفتے شروع ہو رہا ہے، جس میں محصولات اور برآمدی پابندیوں پر اختلافات کم کرنے کی کوشش ہوگی۔ مبصرین کے مطابق بڑی پیش رفت کا امکان کم ہے، لیکن مذاکرات کا جاری رہنا خود ایک مثبت اشارہ ہے۔",
        "یورپی یونین نے ایک ملک پر اقتصادی پابندیوں کے نئے پیکج کا اعلان کیا ہے، جو معاشی دباؤ ڈالنے کا ایک طریقہ ہے۔ اس کے تحت کچھ بینکوں کے اثاثے منجمد اور توانائی کی برآمدات محدود کی جائیں گی۔ متاثرہ حکومت نے اسے ناانصافی قرار دیا اور جوابی اقدامات کی دھمکی دی۔",
    ],
}

GOLD_EASY_META_UR = [
    ("سلامتی کونسل، قرارداد، جنگ بندی", "no"),
    ("تجارتی بات چیت، محصولات، برآمدی پابندیاں", "yes"),
    ("پابندیاں، منجمد اثاثے، جوابی اقدامات", "no"),
]

for d in DEPTHS:
    assert len(GOLD_DEPTH[d]) == len(GOLD_SOURCES), f"GOLD_DEPTH['{d}'] needs one rewrite per GOLD_SOURCES entry"
assert len(GOLD_EASY_META) == len(GOLD_SOURCES), "GOLD_EASY_META needs one entry per GOLD_SOURCES entry"
print("Gold samples OK:", len(GOLD_SOURCES), "depth examples,",
      {l: len(v) for l, v in GOLD_LANG.items()}, "language pairs")

Gold samples OK: 3 depth examples, {'urdish': 4, 'ur': 2} language pairs


## 6. Prompts
Each request is a chat: the rules go in the system message, each gold example is a separate user → assistant turn, and the real article is the last user message.

In [44]:
SYS_PREP = """You summarise Urdu news articles in English.
Reply in exactly this format and nothing else:
SUMMARY: 3 to 5 plain English sentences with the main facts of the article. Keep every name, number and date exactly as in the article; copy each person's name from the article and never guess or merge names. Use standard English spellings of names and titles. Add nothing that is not in the article and do not comment on the article.
TERMS: 1 to 3 political, legal, economic or international-relations concepts from your summary that a beginner would not know (for example: governor's rule, sanctions, constituency, IMF bailout). Never pick ordinary words (protest, gunfire, police, school, injured). Comma-separated, lowercase. Write none if there are none."""

SYS_DEPTH = {
    "easy": """You explain news to a reader who knows nothing about politics, government or international relations.
Rules:
- One paragraph of plain, natural prose with short sentences: one main idea per sentence, most sentences under 20 words.
- Use everyday words instead of formal ones (for example: "ratified" becomes "officially approved", "bilateral" becomes "between the two countries").
- Keep every fact of the Text: every name, place, number and date, with the sayme meaning. Never make a fact stronger or weaker (for example, "delayed" must not become "cancelled"). Do not add facts, guesses or conclusions about the event.
- Explain difficult political, legal, economic or international concept a beginner would not know, including the listed Terms, once, where it first appears: either as a short phrase, or in the very next sentence ("This means ..."). Never put two explanations in one sentence, and never collect explanations at the end. Skip any Term whose everyday words already make its meaning clear.
- Explain difficult IR actor the first time they appear, say briefly who they are, but only if you are sure (for example: "NATO, a military alliance of North American and European countries"). If you are not sure, just use the name. Never explain things almost everyone knows.
- You may add ONE short sentence of general background that helps a beginner understand the main concept (for example: "In many countries, parliament must approve a treaty before it takes effect."). It must be general knowledge, never a new fact about this event.
- If Example is yes, add one short everyday example of one of the Terms. If Example is no, add no example.
- Never spell out an abbreviation unless the Text spells it out.
- Rewrite ONLY the Text in the latest message. Output only the rewrite, with no notes or comments.""",
    "medium": """You explain news to a reader with basic or intermediate knowledge of international relations: they know common terms but not specialist ones. Write in simple, easy-to-read language.
Rules:
- One paragraph of clear, natural prose.
- Use the correct political and legal terms. Explain the some most important/difficult IR or political concepts in the Text, each in a short phrase where it first appears. Do not explain common ones (government, election, protest, minister).
- Identify an actor briefly only where it helps understanding (for example: "the WTO, the body that oversees global trade rules").
- Keep every fact of the Text: every name, place, number and date, with the same meaning. Do not add facts, descriptions, or names of organisations, groups or countries that are not in the Text.
- Never spell out an abbreviation unless the Text spells it out.
- Rewrite ONLY the Text in the latest message. Output only the rewrite, with no notes or comments.""",
    "hard": """You write news for a reader who is familiar with international relations concepts and terms.
Rules:
- One paragraph of clear, simple prose that uses the correct technical IR and legal terms without explaining them. The language must stay easy to read: only the terms are technical, not the sentences language or structure.
- Keep official and legal terms exactly as in the Text; do not swap them for near-synonyms (for example: "postponed" is not "cancelled", "sanctions" is not "embargo", "did not vote" is not "abstained").
- Never add facts, motives, predictions, consequences or interpretations that the Text does not state. Avoid speculative words (potentially, may indicate, likely, signalling, in a move to).
- Precise does not mean inflated: prefer plain verbs (said, led, began, visited) over showy words (spearheaded, delineated, populace, modalities).
- Keep every fact of the Text: every name, place, number and date. About the same length as the Text; never compress it into one or two sentences.
- Never spell out an abbreviation unless the Text spells it out.
- Rewrite ONLY the Text in the latest message. Output only the rewrite, with no notes or comments.""",
}

SYS_LANG = {
        "urdish": """You convert English text to Urdish: Urdu written in English (Roman) letters, mixed with common English words, the way an educated Pakistani writes in WhatsApp messages.
Rules:
- Keep the exact meaning of every sentence. Do not add, drop or change any fact, name or number. If you are not sure of the right Urdu word, keep the English word instead of guessing.
- Match the difficulty of the English. Simple English gets simple everyday words; formal English can use more formal Urdu.
- Sound natural, not translated word by word. You may join two short sentences or change word order, as long as nothing is lost.
- Keep news and technical terms in English where people would actually text them in English (protest, road, supply, security, sanctions, ceasefire, governor's rule). Do not swap them for heavy, bookish Urdu words.
- Keep explanations as explanations (yaani..., jiska matlab hai...).
- Use Pakistani Urdu words, never Hindi words (sarkar, shanti, chinta, vishay, adhikari, samasya, jaanch).
- Spell names exactly as in the English. No Urdu script and no Devanagari. Keep numbers exactly as in the English.
- Convert ONLY the text in the latest message. Output only the converted text.""",
    "ur": """You translate English text into natural, standard Urdu in Urdu script, as written in Pakistani newspapers.
Rules:
- Same content, same order, same difficulty, same explanations. Do not simplify, add or drop anything.
- Use established Urdu terms (پابندیاں، جنگ بندی، سلامتی کونسل). If no Urdu term is common, write the English term in Urdu script.
- Keep numbers exactly as in the English. For names of people and places, use the spelling in the original Urdu article if one is given.
- Write only in Urdu script. Never use Devanagari (Hindi) letters or Hindi words.
- Translate ONLY the English text in the latest message. Output only the Urdu text.""",
}

def depth_user(depth, text, terms="none", ex="no"):
    if depth == "easy":
        return f"Text: {text}\nTerms: {terms}\nExample: {ex}"
    return f"Text: {text}"

def depth_messages(depth, text, terms, ex):
    msgs = [{"role": "system", "content": SYS_DEPTH[depth]}]
    for i, s in enumerate(GOLD_SOURCES):
        t, e = GOLD_EASY_META[i] if depth == "easy" else ("none", "no")
        msgs.append({"role": "user", "content": depth_user(depth, s, t, e)})
        msgs.append({"role": "assistant", "content": GOLD_DEPTH[depth][i]})
    msgs.append({"role": "user", "content": depth_user(depth, text, terms, ex)})
    return msgs

def lang_messages(lang, english, urdu_original=""):
    msgs = [{"role": "system", "content": SYS_LANG[lang]}]
    for en, tr in GOLD_LANG[lang]:
        msgs.append({"role": "user", "content": en})
        msgs.append({"role": "assistant", "content": tr})
    if lang == "ur" and urdu_original:
        # the original article helps with correct Urdu spellings of names, places and terms
        english = (f"Original Urdu article (only for spellings of names, places and terms):\n{urdu_original}\n\n"
                   f"English text to translate:\n{english}")
    msgs.append({"role": "user", "content": english})
    return msgs

# Sentences from the gold answers. If one shows up in an output, the model copied an example.
GOLD_SENTS = {s.strip().lower() for txt in [*GOLD_SOURCES, *sum(GOLD_DEPTH.values(), []),
                                            *[t for p in GOLD_LANG.values() for pair in p for t in pair]]
              for s in re.split(r"[.!?۔]", txt) if len(s.split()) >= 6}

# ---------- Urdu-based pipeline (BASE = "ur") ----------
SYS_PREP_UR = """You summarise Urdu news articles in Urdu.
Reply in exactly this format and nothing else:
SUMMARY: a list of 4 to 7 short facts from the article, one per line, each starting with "- ". Each fact is one short, simple sentence in everyday Urdu (Urdu script). Keep every name, number and date exactly as in the article; copy each person's name from the article and never guess or merge names. Write numbers as digits. Add nothing that is not in the article.
TERMS: 0 to 3 political, legal, economic or international-relations concepts from your summary that a beginner would not know (for example: گورنر راج، پابندیاں، حلقہ انتخاب). Write each exactly as it appears in your summary. Never pick ordinary words (احتجاج، پولیس، زخمی). Separate them with commas. Write none if there are none."""

UR_LINE = ("- Write in simple, natural Urdu (Urdu script), the way an educated Pakistani explains news in conversation, not in heavy newspaper style. "
           "Prefer common spoken words (for example: شروع instead of آغاز، مدد instead of معاونت، اس کے علاوہ instead of علاوہ ازیں). "
           "English words that are common in everyday Urdu are fine. The depth changes how much IR knowledge is assumed, never how difficult the language is. Write numbers as digits.\n"
           "- The Text is a list of facts. Turn it into one connected paragraph for this reader, in your own words. Do not copy the facts line by line.\n")
SYS_DEPTH_UR = {d: SYS_DEPTH[d].replace("- Rewrite ONLY the Text", UR_LINE + "- Rewrite ONLY the Text") for d in DEPTHS}
SYS_DEPTH_UR["easy"] = SYS_DEPTH_UR["easy"].replace(UR_LINE, UR_LINE + "- Use simple, everyday spoken Urdu words, not bookish ones. If the Text uses a formal newspaper word, say it in everyday words instead.\n")
def as_facts(text):
    """Turn a prose gold source into a fact list, so gold inputs look like real inputs."""
    return "\n".join("- " + s.strip() + "۔" for s in text.split("۔") if s.strip())
    
def depth_messages_ur(depth, text, terms, ex):
    msgs = [{"role": "system", "content": SYS_DEPTH_UR[depth]}]
    for i, s in enumerate(GOLD_SOURCES_UR):
        t, e = GOLD_EASY_META_UR[i] if depth == "easy" else ("none", "no")
        msgs.append({"role": "user", "content": depth_user(depth, as_facts(s), t, e)})
        msgs.append({"role": "assistant", "content": GOLD_DEPTH_UR[depth][i]})
    msgs.append({"role": "user", "content": depth_user(depth, text, terms, ex)})
    return msgs


READER = {
        "easy":   "a reader with no IR knowledge: use simple, everyday words and short sentences; describe actions in plain words instead of formal nouns",
    "medium": "a reader with some IR knowledge: use simple, clear words",
    "hard":   "a reader familiar with IR terms: keep the technical terms, but use clear, simple sentences",
}

SYS_UR_TO_EN = """You translate Urdu text (Urdu script) into clear, natural English.
Rules:
- Same content, same order, same explanations. Do not add or drop anything.
- Write for the Reader given in the message. Never make the English more formal than needed: prefer plain words (for example: "began" rather than "commenced", "use" rather than "utilize").
- Keep every name, number and date. Use standard English spellings of names and places.
- Keep the exact meaning of official actions (for example: "appointed" is not "elected", "suspended" is not "dismissed"), and never add names or details that are not in the Urdu text.
- Output only the English translation."""

GOLD_UR_EN = [(GOLD_DEPTH_UR["easy"][0], GOLD_DEPTH["easy"][0], "easy"),
              (GOLD_DEPTH_UR["medium"][1], GOLD_DEPTH["medium"][1], "medium"),
              (GOLD_DEPTH_UR["hard"][2], GOLD_DEPTH["hard"][2], "hard")]

def ur_to_en_messages(urdu, depth):
    msgs = [{"role": "system", "content": SYS_UR_TO_EN}]
    for ur, en, d in GOLD_UR_EN:
        msgs.append({"role": "user", "content": f"Reader: {READER[d]}\n\n{ur}"})
        msgs.append({"role": "assistant", "content": en})
    msgs.append({"role": "user", "content": f"Reader: {READER[depth]}\n\n{urdu}"})
    return msgs

SYS_URDISH_FROM_UR = """You write Urdish: Urdu written in English (Roman) letters, mixed with common English words, the way an educated Pakistani writes in WhatsApp messages.
You get an Urdu text and its English translation. They say the same thing.
Rules:
- Follow the Urdu text sentence by sentence. Do not add, drop or change any fact, name or number.
- Write for the Reader given in the message, the way people actually text: where the Urdu uses a formal or bookish word, use the everyday word or the common English word instead (for example: log instead of afrad, shuru instead of aaghaz, madad instead of muawanat).
- Keep news and technical terms in English the way people text them (protest, government, sanctions, ceasefire, governor's rule).
- Use Pakistani Urdu words, never Hindi words (sarkar, shanti, chinta, vishay, adhikari, samasya, jaanch).
- Spell names exactly as in the English translation. No Urdu script and no Devanagari. Keep numbers as digits.
- Output only the Urdish text."""

# (English, Urdu, Urdish, depth) built from the existing gold examples
GOLD_UR_URDISH = [
    (GOLD_LANG["ur"][0][0], GOLD_LANG["ur"][0][1], GOLD_LANG["urdish"][0][1], "medium"),
    (GOLD_LANG["ur"][1][0], GOLD_LANG["ur"][1][1], GOLD_LANG["urdish"][2][1], "easy"),
]

def urdish_from_ur_user(urdu, english, depth):
    return f"Reader: {READER[depth]}\n\nUrdu text:\n{urdu}\n\nEnglish translation:\n{english}"

def urdish_from_ur_messages(urdu, english, depth="medium"):
    msgs = [{"role": "system", "content": SYS_URDISH_FROM_UR}]
    for en, ur, rd, d in GOLD_UR_URDISH:
        msgs.append({"role": "user", "content": urdish_from_ur_user(ur, en, d)})
        msgs.append({"role": "assistant", "content": rd})
    msgs.append({"role": "user", "content": urdish_from_ur_user(urdu, english, depth)})
    return msgs
# also detect copying of the Urdu gold examples
GOLD_SENTS |= {s.strip().lower() for txt in [*GOLD_SOURCES_UR, *sum(GOLD_DEPTH_UR.values(), [])]
               for s in re.split(r"[.!?۔]", txt) if len(s.split()) >= 6}


print("Prompts ready")

Prompts ready


## 7. Load the articles folder
Reads every `.jsonl`, `.json` and `.txt` file under `ARTICLES_DIR`, including the output of `scrape_urdu_articles.py`.

In [45]:
def make_article_id(url, text):
    key = url.strip() if url else text.strip()[:2000]
    return hashlib.sha256(key.encode("utf-8")).hexdigest()

def _as_article(d, fallback_url):
    text = d.get("text") or d.get("content") or d.get("body") or ""
    url = d.get("url") or d.get("link") or fallback_url
    return {"url": url, "text": text, "title": d.get("title", "")}

def load_articles(folder):
    arts = []
    files = sorted(glob.glob(os.path.join(folder, "**", "*"), recursive=True))
    for path in files:
        ext = os.path.splitext(path)[1].lower()
        rel = os.path.relpath(path, folder)
        try:
            if ext == ".jsonl":
                for n, line in enumerate(open(path, encoding="utf-8")):
                    if line.strip():
                        arts.append(_as_article(json.loads(line), f"{rel}#{n}"))
            elif ext == ".json":
                data = json.load(open(path, encoding="utf-8"))
                items = data if isinstance(data, list) else [data]
                for n, d in enumerate(items):
                    arts.append(_as_article(d, f"{rel}#{n}"))
            elif ext == ".txt":
                arts.append({"url": rel, "text": open(path, encoding="utf-8").read(), "title": ""})
        except Exception as e:
            print(f"  skipped {rel}: {type(e).__name__}: {e}")
    out, seen = [], set()
    for a in arts:
        if not a["text"].strip():
            continue
        a["article_id"] = make_article_id(a["url"], a["text"])
        if a["article_id"] in seen:          # same article twice in THIS folder: process once
            continue
        seen.add(a["article_id"])
        if NUM_SHARDS > 1 and int(a["article_id"][:8], 16) % NUM_SHARDS != SHARD_INDEX:
            continue
        out.append(a)
    return out[:MAX_ARTICLES] if MAX_ARTICLES else out

assert os.path.isdir(ARTICLES_DIR), f"Folder not found: {ARTICLES_DIR}"
ARTICLES = load_articles(ARTICLES_DIR)
print(f"{len(ARTICLES)} articles loaded for machine {MACHINE_ID}")
if ARTICLES:
    a = ARTICLES[0]
    print("First one:", a["url"], "|", a["text"][:200].replace("\n", " "), "...")



40 articles loaded for machine bisma_test
First one: https://www.bbc.com/urdu/articles/cmp3gn5nrpddo | پاکستان کے صوبہ خیبرپختونخوا میں ایمرجنسی اور گورنر راج کے خلاف ضلع خیبر کے علاقے باڑہ میں گذشتہ روز (13 اکتوبر) ہونے والے پُرتشدد مظاہروں کے بعد آج (بدھ) بھی بازار بند رہے جبکہ اس علاقے میں پولیس اور ...


## 8. LLM helpers

In [46]:
LOCK = threading.Lock()

def clean(t):
    t = re.sub(r"<think>.*?</think>", "", t or "", flags=re.S).strip()
    t = re.sub(r"^\s*(Urdish|Urdu|English|Rewrite|Output|Text)\s*:\s*", "", t, flags=re.I)
    t = t.replace("*", "")
    return re.sub(r"\s*\n+\s*", " ", t.strip().strip('"')).strip()

def llm(messages, max_tokens=900, model=None):
    model = model or MODEL
    mt = max_tokens * 8 if is_reasoning(model) else max_tokens
    for attempt in range(5):
        try:
            r = client.chat.completions.create(model=model, max_tokens=mt, temperature=TEMPERATURE,
                                               messages=messages)
            c = r.choices[0].message.content or ""
            if not c.strip():
                print(f"   [empty] {model} | finish={r.choices[0].finish_reason} | usage={getattr(r, 'usage', None)}")
            return c
            
        except Exception as e:
            print(f"   [llm retry {attempt+1}] {type(e).__name__}: {str(e)[:80]}")
            time.sleep(2 ** attempt)
    raise RuntimeError("llm failed after retries")

def copied_gold(t):
    tl = t.lower()
    return [s for s in GOLD_SENTS if s in tl]

def remove_gold(t):
    """Last resort: drop any copied example sentence from the output."""
    parts = re.split(r"(?<=[.!?۔])\s+", t)
    return " ".join(p for p in parts if not copied_gold(p)).strip()

DEVANAGARI = re.compile(r"[\u0900-\u097F]")
HINDI_WORDS = re.compile(r"\b(sarkar|shanti|shant|vishay|adhikari|hinsak|hinsa|samasya|upyog|rashtra|suraksha|yuddh|mantri|pradhan|chinta|jaanch|kripya|dhanyavad)\b", re.I)
def problem(t, kind, source=""):
    """What is wrong with an output, or None. Used only to ask the model ONE more time; nothing is ever rejected."""
    if not t:
        return "Your answer was empty."
    if copied_gold(t):
        return "You copied the earlier examples. Give the answer for my latest text only."
    if re.search(r"\bnote\s*:|\b(is|are) not (present|mentioned|found|given)\b|\bthe (text|summary) (says|does|mentions|states)", t, re.I):
        return "Your answer contains a comment about the text. Output only the rewrite itself."
    if kind in ("easy", "medium", "hard") and source:
        dig = str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789")
        extra = set(re.findall(r"[0-9]+", t.translate(dig))) - set(re.findall(r"[0-9]+", source.translate(dig)))
        extra = {x for x in extra if len(x) > 1}   # single digits are often written as words in the source
        if extra:
            return f"Your answer contains numbers that are not in the Text ({', '.join(sorted(extra))}). Remove anything that is not in the Text."
    if DEVANAGARI.search(t):
        return "You used Hindi (Devanagari) letters. Write it again without any Devanagari."
    if kind == "urdish":
        words = re.findall(r"[a-z']+", t.lower())
        urdu_words = {"hai", "hain", "ka", "ki", "ke", "mein", "ne", "ko", "se", "aur", "tha", "thi", "thay", "hua", "hue", "kiya", "gaya", "gayi", "nahi", "bhi", "yeh", "woh"}
        if words and sum(w in urdu_words for w in words) / len(words) < 0.08:
            return "Your answer is in English. Write it in Urdish: Urdu sentences in Roman letters, with only some English words mixed in."
        if source and len(t.split()) < 0.7 * len(source.split()):
            return "You left out some sentences. Convert every sentence of the Urdu text."
    if kind in ("medium", "hard") and source and len(t.split()) < 0.6 * len(source.split()):
        return "Your answer left out facts. Cover every sentence of the Text, with every name, place and number."
    if kind == "easy" and source and len(t.split()) < 0.8 * len(source.split()):
        return "Your answer left out facts. Cover every sentence of the Text, with every name, place and number."
    return None

def generate(messages, kind, source="", model=None, max_tokens=900):
    """One call; if something is clearly wrong, ask once more with the reason. The output is always kept."""
    t = clean(llm(messages, max_tokens, model))
    why = problem(t, kind, source)
    if why:
        print(f"   [retry {kind}] {why[:90]}")
        t2 = clean(llm(messages + [{"role": "assistant", "content": t or "(empty)"},
                                    {"role": "user", "content": why}], max_tokens, model))
        still = problem(t2, kind, source) if t2 else "empty"
        if still:
            print(f"   [still wrong {kind}] {still[:90]}")
        if t2 and (still is None or not t):
            t = t2
    return remove_gold(t) if copied_gold(t) else t

def parse_prep(t):
    t = re.sub(r"<think>.*?</think>", "", t or "", flags=re.S).replace("**", "")
    m = re.search(r"SUMMARY\s*:\s*(.*?)(?:\n\s*TERMS\s*:|$)", t, re.S | re.I)
    summary = clean(m.group(1) if m else t)
    m = re.search(r"TERMS\s*:\s*(.*)", t, re.I)
    terms = [x.strip(" .'\"").lower() for x in (m.group(1) if m else "").split(",")]
    ORDINARY = {"protest", "protests", "gunfire", "police", "injured", "school", "schools", "road", "violence"}
    terms = [x for x in terms if x and x not in ("none", "n/a") and x.isascii()
             and x not in ORDINARY and x in summary.lower()][:3]
    return summary, terms

def parse_prep_ur(t):
    t = re.sub(r"<think>.*?</think>", "", t or "", flags=re.S).replace("**", "")
    m = re.search(r"SUMMARY\s*:\s*(.*?)(?:\n\s*TERMS\s*:|$)", t, re.S | re.I)
    raw = (m.group(1) if m else t).replace("*", "")
    summary = "\n".join(l.strip() for l in raw.strip().splitlines() if l.strip())
    m = re.search(r"TERMS\s*:\s*(.*)", t, re.I)
    terms = [x.strip(" .۔'\"") for x in re.split(r"[،,]", m.group(1) if m else "")]
    terms = [x for x in terms if x and x.lower() not in ("none", "n/a", "کوئی نہیں") and x in summary][:3]
    return summary, terms
    
def chunk(text, max_words=350):
    """Article body without junk lines, cut to max_words."""
    paras = [p.strip() for p in text.split("\n") if p.strip() and not re.search(r"BBC|بی بی سی|©|\.com", p)]
    words = " ".join(paras).split()
    return " ".join(words[:max_words]) if len(words) >= 30 else None

print("Helpers ready")

Helpers ready


## 9. Pipeline: one article → 9 cells
1. Summarise the Urdu article in English and pick the terms to explain.
2. Write the English cell at each depth from the summary.
3. Convert each English cell to Urdish and Urdu.

No filters: an article is only skipped if it has no text, or if the API keeps failing (those are retried on the next run).

In [50]:
PREP_CACHE = {}
if os.path.exists(PREP_PATH):
    for line in open(PREP_PATH, encoding="utf-8"):
        r = json.loads(line)
        PREP_CACHE[r.pop("article_id")] = r

def prep(art):
    aid = art["article_id"]
    if aid in PREP_CACHE:
        return PREP_CACHE[aid]
    body = chunk(art["text"])
    if not body:
        return {"reject": "no_text"}
    if BASE == "ur":
        summary, terms = parse_prep_ur(llm([{"role": "system", "content": SYS_PREP_UR},
                                            {"role": "user", "content": body}], 1600, model=UR_MODEL))
    else:
        summary, terms = parse_prep(llm([{"role": "system", "content": SYS_PREP},
                                         {"role": "user", "content": body}], 600, model=LANG_MODEL))
    if not summary.strip():
        return {"reject": "empty_summary"}
    res = {"summary": summary, "terms": terms}
    with LOCK:
        PREP_CACHE[aid] = res
        with open(PREP_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps({"article_id": aid, **res}, ensure_ascii=False) + "\n")
    return res

def make_ur(art):
    """Urdu summary -> Urdu depths -> English (from Urdu) -> Urdish (from Urdu + English)."""
    base = {"article_id": art["article_id"], "url": art["url"], "machine": MACHINE_ID}
    p = prep(art)
    if p.get("reject"):
        return {**base, "reject": p["reject"]}
    use_ex = int(art["article_id"][:8], 16) % 100 < EXAMPLE_RATE * 100
    row = {**base, "input": p["summary"], "terms": p["terms"], "use_example": use_ex, "cells": {}, "reject": None}
    row["source"] = chunk(art["text"]) or ""
    for depth in DEPTHS:
        ur = generate(depth_messages_ur(depth, p["summary"], "، ".join(p["terms"]) or "none", "yes" if use_ex else "no"),
                      kind=depth, source=p["summary"], model=UR_MODEL, max_tokens=1400)
        if not ur:
            return {**base, "reject": f"empty_ur_{depth}"}
        en = generate(ur_to_en_messages(ur, depth), kind="en", model=UR_MODEL, max_tokens=1100)
        if not en:
            return {**base, "reject": f"empty_en_{depth}"}
        row["cells"][f"ur_{depth}"] = ur
        row["cells"][f"en_{depth}"] = en
        row["cells"][f"urdish_{depth}"] = generate(urdish_from_ur_messages(ur, en, depth), kind="urdish",
                                                   source=en, model=UR_MODEL, max_tokens=1100)
    return row

def make(art):
    if BASE == "ur":
        return make_ur(art)
    base = {"article_id": art["article_id"], "url": art["url"], "machine": MACHINE_ID}
    p = prep(art)
    if p.get("reject"):
        return {**base, "reject": p["reject"]}
    use_ex = int(art["article_id"][:8], 16) % 100 < EXAMPLE_RATE * 100
    row = {**base, "input": p["summary"], "terms": p["terms"], "use_example": use_ex, "cells": {}, "reject": None}
    for depth in DEPTHS:
        row["cells"][f"en_{depth}"] = generate(
            depth_messages(depth, p["summary"], ", ".join(p["terms"]) or "none", "yes" if use_ex else "no"),
            kind=depth, source=p["summary"])
        if not row["cells"][f"en_{depth}"]:
            return {**base, "reject": f"empty_en_{depth}"}
    urdu_original = " ".join((chunk(art["text"]) or "").split()[:250])
    for depth in DEPTHS:
        for lang in ("urdish", "ur"):
            row["cells"][f"{lang}_{depth}"] = generate(
                lang_messages(lang, row["cells"][f"en_{depth}"], urdu_original),
                kind=lang, model=LANG_MODEL, max_tokens=1400 if lang == "ur" else 1100)
    if URDISH_FROM_UR:
        for depth in DEPTHS:
            row["cells"][f"urdishU_{depth}"] = generate(
                urdish_from_ur_messages(row["cells"][f"ur_{depth}"], row["cells"][f"en_{depth}"]),
                kind="urdish", model=LANG_MODEL, max_tokens=1100)
    return row

print("Pipeline ready | BASE =", BASE, "| UR_MODEL =", UR_MODEL if BASE == "ur" else "-")

Pipeline ready | BASE = ur | UR_MODEL = deepseek-ai/DeepSeek-V4.1-Flash


## 10. Pilot: run a couple of articles and read the output
Nothing is saved here. Check the summary and all 9 cells before the full run.

In [51]:
PILOT_IDS = [14,0]
for art in [ARTICLES[i] for i in PILOT_IDS]:
    row = make(art)
    print("=" * 100)
    print("URL:", row["url"], "| skipped:", row.get("reject"))
    if row.get("input"):
        print("\nSUMMARY:", row["input"], "\nTERMS:", row["terms"], "| example:", row["use_example"])
        for d in DEPTHS:
            for l in LANGS:
                print(f"\n[{l} / {d}]\n{row['cells'][f'{l}_{d}']}")
            if f"urdishU_{d}" in row["cells"]:
                print(f"\n[urdish FROM URDU / {d}]\n{row['cells'][f'urdishU_{d}']}")

URL: https://www.bbc.com/urdu/articles/cr1exl07pe60o | skipped: None

SUMMARY: - یمن کی جنگ کے اثرات سعودی عرب کے اندر بھی محسوس کیے جا رہے ہیں۔
- سعودی حکام کے مطابق گذشتہ 3 روز میں حوثیوں نے ریاض اور ابہا کے بین الاقوامی ہوائی اڈوں کو میزائلوں اور ڈرونز سے نشانہ بنایا۔
- ان حملوں میں پائلٹ سمیت 3 افراد ہلاک اور درجنوں زخمی ہوئے۔
- ریاض کے شاہ خالد انٹرنیشنل ایئرپورٹ پر حملے میں سعودیہ کے پائلٹ حمود علی الکلثامی ہلاک ہوئے۔
- پی آئی اے سمیت متعدد بین الاقوامی فضائی کمپنیوں نے ریاض کے لیے پروازیں معطل یا منسوخ کر دیں۔
- گاکا کے مطابق جمعرات کو شاہ خالد انٹرنیشنل ایئرپورٹ پر 2 حملے کیے گئے۔
- حوثی فوج کے ترجمان یحییٰ السریع نے سعودی فضائی حدود کو اپنی فوجی کارروائیوں کا مرکز قرار دیا اور ایک اور میزائل حملے کی ذمہ داری قبول کی۔ 
TERMS: ['فضائی حدود'] | example: True

[en / easy]
The effects of the war in Yemen are also being felt inside Saudi Arabia. According to Saudi officials, in the past 3 days, the Houthis targeted the international airports in Riyadh and Abha with missiles and dron

### 10b. Which model writes the best Urdu / Urdish?
Converts one English cell with a few W&B models side by side. Read them (or ask a native speaker), then set `LANG_MODEL` in section 3 to the winner and re-run sections 3, 4 and 9 before the full run.

In [36]:
CANDIDATES = ["deepseek-ai/DeepSeek-V3.1", "google/gemma-4-31B-it", "moonshotai/Kimi-K2.6",
              "Qwen/Qwen3.8-27B", "meta-llama/Llama-3.3-70B-Instruct","deepseek-ai/DeepSeek-V4.1-Flash"]
art = ARTICLES[0]
p = prep(art)
english = generate(depth_messages("easy", p["summary"], ", ".join(p["terms"]) or "none", "no"),
                   kind="easy", source=p["summary"])
original = " ".join((chunk(art["text"]) or "").split()[:250])
print("ENGLISH (medium):\n", english)
for m in CANDIDATES:
    for lang in ("urdish", "ur"):
        try:
            out = generate(lang_messages(lang, english, original), kind=lang, model=m, max_tokens=1400)
        except Exception as e:
            out = f"(failed: {type(e).__name__})"
        print(f"\n----- {m} / {lang}\n{out}")

ENGLISH (medium):
 There are protests in Pakistan's Khyber Pakhtunkhwa province. The protests are against the emergency, which is a special situation where the government has more control. They are also against the governor's rule, which means the federal government runs the province directly. The protests are in the Bara area of Khyber district. Markets are closed. There are many police and FC, which is a security force, in the area. On 13 October, police and PTI workers clashed. PTI workers are supporters of a political party. 21 people were hurt, including police. 62 people were taken into custody. The protests are in the area where Suhail Afridi was in charge. He was the Chief Minister, but he was removed. At least five people with gunshot wounds were taken to the hospital. Gunshot wounds are injuries from gunfire, which is when a gun is fired. CCPO Peshawar Dr Mian Saeed visited Bara. He checked the security arrangements. He asked people to remain peaceful.

----- deepseek-ai/Deep

## 11. Full run (resumable)
Every processed article is appended to `grid_raw_v2_<machine>.jsonl`. If the session dies, run this cell again and it continues.

In [ ]:
done = {}
if os.path.exists(RAW_PATH):
    for line in open(RAW_PATH, encoding="utf-8"):
        r = json.loads(line)
        if not str(r.get("reject") or "").startswith(("error", "empty")):   # API errors and empty outputs are retried   # API errors are retried
            done[r["article_id"]] = r

todo = [a for a in ARTICLES if a["article_id"] not in done]
print(f"{len(ARTICLES)} articles | {len(done)} already done | {len(todo)} to process | model = {MODEL}")

def safe(art):
    try:
        return make(art)
    except Exception as e:
        return {"article_id": art["article_id"], "url": art["url"], "machine": MACHINE_ID,
                "reject": f"error:{type(e).__name__}"}

B = WORKERS * 4
t0 = time.time()
with ThreadPoolExecutor(WORKERS) as ex, open(RAW_PATH, "a", encoding="utf-8") as f:
    for i in range(0, len(todo), B):
        for r in ex.map(safe, todo[i:i + B]):
            done[r["article_id"]] = r
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
        f.flush()
        counts = collections.Counter(r.get("reject") or "done" for r in done.values())
        print(f"  {min(i + B, len(todo))}/{len(todo)} | {time.time() - t0:.0f}s | " +
              ", ".join(f"{k}={v}" for k, v in counts.most_common()))

## 12. Export this machine's dataset
Writes in `OUT_DIR`:
- `grid_<machine>.csv`: every article as 9 rows (one per language × depth). The adapter notebook merges these.
- `review_<machine>.csv`: about 10% of articles (all 9 cells) for a native speaker to read, with `ok` / `notes` columns.

In [ ]:
write_csv(grid_path,
          ["article_id", "url", "machine", "language", "depth", "input", "facts", "output", "terms", "use_example"],
          [[r["article_id"], r["url"], r["machine"], l, d, r.get("source") or r["input"], r["input"],
            r["cells"][f"{l}_{d}"], "; ".join(r["terms"]), r["use_example"]]
           for r in good for d in DEPTHS for l in LANGS])

rows = {}
for line in open(RAW_PATH, encoding="utf-8"):
    r = json.loads(line)
    rows[r["article_id"]] = r
good = [r for r in rows.values()
        if not r.get("reject") and all(r["cells"].get(f"{l}_{d}") for l in LANGS for d in DEPTHS)]
skipped = collections.Counter(r.get("reject") for r in rows.values() if r.get("reject"))

grid_path = os.path.join(OUT_DIR, f"grid_{MACHINE_ID}.csv")
write_csv(grid_path,
          ["article_id", "url", "machine", "language", "depth", "input", "output", "terms", "use_example"],
          [[r["article_id"], r["url"], r["machine"], l, d, r["input"], r["cells"][f"{l}_{d}"],
            "; ".join(r["terms"]), r["use_example"]]
           for r in good for d in DEPTHS for l in LANGS])

sample = random.Random(7).sample(good, min(len(good), max(5, round(len(good) * 0.1))))
write_csv(os.path.join(OUT_DIR, f"review_{MACHINE_ID}.csv"),
          ["article_id", "language", "depth", "input", "output", "ok (y/n)", "notes"],
          [[r["article_id"], l, d, r["input"], r["cells"][f"{l}_{d}"], "", ""]
           for r in sample for d in DEPTHS for l in LANGS])

print(f"Articles in dataset: {len(good)} -> {len(good) * 9} rows in {grid_path}")
print("Skipped:", dict(skipped) or "none")